# Phase 2B. LSTM 딥러닝 모델 구축 및 Random Forest 비교

## 목표
- 시계열 흐름(20일)을 학습하는 LSTM 모델 구축
- Random Forest(정적 패턴) vs LSTM(동적 흐름) 성능 비교
- GPU(RTX 4070) 활용 학습

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data_loader import download_all
from src.features   import process_all, build_combined_dataset
from src.model      import split_data, train_model, evaluate_model, FEATURE_COLS

# GPU 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")
print(f"GPU 이름: {torch.cuda.get_device_name(0)}")

# 데이터 로드
stock_data = download_all(save_csv=False)
processed  = process_all(stock_data, save_csv=False)
combined   = build_combined_dataset(processed)
print("데이터 로드 완료")

## 1. 시퀀스 데이터 구성

Random Forest는 하루치 데이터를 입력으로 받지만
LSTM은 N일치 데이터를 순서대로 입력받는다.
```
look_back = 20 (20일치 흐름을 보고 다음날 예측)

입력 Shape: (샘플 수, 20일, 7개 Feature)
출력 Shape: (샘플 수, 1)  ← 0 또는 1
```

In [ ]:
LOOK_BACK = 20  # 20일치 흐름을 보고 예측

def create_sequences(df: pd.DataFrame, look_back: int = 20):
    """
    시계열 데이터를 LSTM 입력용 시퀀스로 변환한다.
    
    예시)
    look_back = 3 이면
    [Day1, Day2, Day3] → Day4 예측
    [Day2, Day3, Day4] → Day5 예측
    ...
    """
    X, y = [], []
    
    data   = df[FEATURE_COLS].values
    target = df["Target"].values
    
    for i in range(look_back, len(df)):
        X.append(data[i-look_back:i])  # 과거 20일
        y.append(target[i])            # 다음날 타겟
    
    return np.array(X), np.array(y)


# 종목별로 시퀀스 생성 (종목 경계에서 섞이지 않게)
X_list, y_list = [], []

for ticker in ["NVDA", "TSM", "AMD", "ASML", "QCOM"]:
    df_ticker = combined[combined["Ticker"] == ticker].copy()
    X_seq, y_seq = create_sequences(df_ticker, LOOK_BACK)
    X_list.append(X_seq)
    y_list.append(y_seq)
    print(f"[{ticker}] 시퀀스 shape: {X_seq.shape}")

X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)

print(f"\n전체 시퀀스 shape: {X_all.shape}")
print(f"타겟 shape: {y_all.shape}")
print(f"타겟 분포: 상승={y_all.sum()} / 하락={len(y_all)-y_all.sum()}")

In [ ]:
# 시계열 분리 (앞 80% 학습 / 뒤 20% 테스트)
split_idx = int(len(X_all) * 0.8)

X_train = X_all[:split_idx]
X_test  = X_all[split_idx:]
y_train = y_all[:split_idx]
y_test  = y_all[split_idx:]

print(f"학습 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")

# 정규화 (2D로 변환 후 스케일링 → 다시 3D로)
scaler = StandardScaler()

n_train, n_steps, n_features = X_train.shape
n_test = X_test.shape[0]

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(n_train, n_steps, n_features)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(n_test, n_steps, n_features)

print("정규화 완료")

In [ ]:
class StockDataset(Dataset):
    """
    PyTorch Dataset 클래스
    DataLoader가 배치(batch) 단위로 데이터를 불러올 수 있게 해줌
    """
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64

train_dataset = StockDataset(X_train_scaled, y_train)
test_dataset  = StockDataset(X_test_scaled,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"학습 배치 수: {len(train_loader)}")
print(f"테스트 배치 수: {len(test_loader)}")

## 2. LSTM 모델 구조
```
입력 (20일 × 7 Feature)
        ↓
LSTM Layer (hidden_size=64, num_layers=2)
        ↓
Dropout (0.3) ← 과적합 방지
        ↓
Fully Connected Layer (64 → 1)
        ↓
Sigmoid ← 확률값 (0~1) 출력
```

### 주요 파라미터
- hidden_size : LSTM 내부 메모리 크기
- num_layers  : LSTM 레이어 수 (깊을수록 복잡한 패턴 학습)
- dropout     : 과적합 방지 (학습 중 랜덤하게 뉴런 비활성화)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, 
                 num_layers=2, dropout=0.3):
        super(LSTMModel, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout,
            batch_first = True    # (batch, seq, feature) 순서
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # LSTM 통과
        lstm_out, _ = self.lstm(x)
        
        # 마지막 시점의 출력만 사용
        last_output = lstm_out[:, -1, :]
        
        # Dropout → FC → Sigmoid
        out = self.dropout(last_output)
        out = self.fc(out)
        out = self.sigmoid(out)
        
        return out.squeeze()


# 모델 생성 및 GPU 이동
model_lstm = LSTMModel(
    input_size  = n_features,
    hidden_size = 64,
    num_layers  = 2,
    dropout     = 0.3
).to(device)

print(model_lstm)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model_lstm.parameters()):,}")

In [ ]:
# 손실함수 & 옵티마이저
criterion = nn.BCELoss()                                    # Binary Cross Entropy
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5, verbose=True        # 성능 정체 시 lr 감소
)

EPOCHS = 50

train_losses, val_losses = [], []

print("🚀 LSTM 학습 시작 (GPU: RTX 4070)")
print("=" * 50)

for epoch in range(EPOCHS):
    # ── 학습 ──────────────────────────────────────────────
    model_lstm.train()
    train_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        output = model_lstm(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # ── 검증 ──────────────────────────────────────────────
    model_lstm.eval()
    val_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            output  = model_lstm(X_batch)
            loss    = criterion(output, y_batch)
            val_loss += loss.item()
    
    train_loss /= len(train_loader)
    val_loss   /= len(test_loader)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    scheduler.step(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f}")

print("=" * 50)
print("학습 완료")

In [ ]:
# ── LSTM 예측 ────────────────────────────────────────────────
model_lstm.eval()
all_probs = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        probs   = model_lstm(X_batch)
        all_probs.extend(probs.cpu().numpy())

y_prob_lstm = np.array(all_probs)
y_pred_lstm = (y_prob_lstm >= 0.5).astype(int)

acc_lstm = accuracy_score(y_test, y_pred_lstm)
auc_lstm = roc_auc_score(y_test, y_prob_lstm)

print(f"LSTM 정확도 : {acc_lstm:.4f}")
print(f"LSTM AUC    : {auc_lstm:.4f}")

# ── Random Forest 결과 (비교용) ──────────────────────────────
X_train_rf, X_test_rf, y_train_rf, y_test_rf = split_data(combined)
model_rf   = train_model(X_train_rf, y_train_rf)
results_rf = evaluate_model(model_rf, X_test_rf, y_test_rf)

# ── 최종 비교 표 ─────────────────────────────────────────────
comparison = pd.DataFrame({
    "모델"      : ["Phase 1 (RF, 7 Features)",
                   "Phase 2A (RF + PCA, 4 주성분)",
                   "Phase 2B (LSTM, 20일 시퀀스)"],
    "Accuracy" : [results_rf["accuracy"], 0.5114, acc_lstm],
    "AUC"      : [results_rf["auc"],      0.5186, auc_lstm],
})

print("\n전체 모델 성능 비교")
display(comparison.round(4))

# ── 학습 곡선 시각화 ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 학습/검증 손실
axes[0].plot(train_losses, label="Train Loss", color="blue")
axes[0].plot(val_losses,   label="Val Loss",   color="red")
axes[0].set_title("LSTM 학습 곡선", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 모델별 AUC 비교
models = ["RF\n(Phase 1)", "RF+PCA\n(Phase 2A)", "LSTM\n(Phase 2B)"]
aucs   = [results_rf["auc"], 0.5186, auc_lstm]
colors = ["#3498db", "#e74c3c", "#2ecc71"]

bars = axes[1].bar(models, aucs, color=colors, edgecolor="white", width=0.5)
axes[1].axhline(y=0.5, color="black", linestyle="--", alpha=0.5, label="랜덤 기준선")
axes[1].set_title("모델별 AUC 비교", fontsize=13, fontweight="bold")
axes[1].set_ylabel("AUC Score")
axes[1].set_ylim(0.48, 0.60)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f"{val:.4f}",
                 ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Phase 1 vs 2A vs 2B 최종 비교", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("../results/figures/final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()